In [16]:
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
from collections import deque
import matplotlib.pyplot as plt
import cv2
from torchsummary import summary

In [2]:
# Hyperparameters

GAMMA = 0.99
EPSILON_START = 1.0
EPSILON_END = 0.1
EPSILON_DECAY = 1000
LEARNING_RATE = 0.0003
BATCH_SIZE = 64
TARGET_UPDATE = 1000
MEMORY_SIZE = 100000
NUM_EPISODES = 300

In [5]:
class OnlineDQN(nn.Module):
    """
    """

    def __init__(self, input_shape, num_actions):
        super(OnlineDQN, self).__init__()
        self.fc1 = nn.Linear(input_shape, 512)
        self.fc2 = nn.Linear(512, num_actions)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return x

In [6]:
class TargetDQN(nn.Module):
    """
    """

    def __init__(self, input_shape, num_actions):
        super(TargetDQN, self).__init__()
        self.fc1 = nn.Linear(input_shape, 512)
        self.fc2 = nn.Linear(512, num_actions)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return x

In [3]:
class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        state, action, reward, next_state, done = zip(*random.sample(self.buffer, batch_size))
        return (np.array(state), np.array(action), np.array(reward),
                np.array(next_state), np.array(done))

    def __len__(self):
        return len(self.buffer)

In [7]:
def select_action(state, epsilon, n_actions, policy_net, device):
    if random.random() > epsilon:
        with torch.no_grad():
            state = torch.FloatTensor(state).unsqueeze(0).to(device)
            return policy_net(state).argmax(1).item()
    else:
        return random.randrange(n_actions)


In [8]:
def optimize_model(online_dqn: OnlineDQN ,memory: ReplayBuffer, device):
    if len(memory) < BATCH_SIZE:
        return
    state, action, reward, next_state, done = memory.sample(BATCH_SIZE)

    state = torch.FloatTensor(state).to(device)
    action = torch.LongTensor(action).to(device)
    reward = torch.FloatTensor(reward).to(device)
    next_state = torch.FloatTensor(next_state).to(device)
    done = torch.FloatTensor(done).to(device)

    q_values = online_dqn(state)
    next_q_values = online_dqn(next_state)
    next_q_state_values = online_dqn(next_state)

    q_value = q_values.gather(1, action.unsqueeze(1)).squeeze(1)
    next_q_value = next_q_state_values.gather(1, next_q_values.argmax(1).unsqueeze(1)).squeeze(1)
    expected_q_value = reward + GAMMA * next_q_value * (1 - done)

    loss = (q_value - expected_q_value.detach()).pow(2).mean()

    online_dqn.optimizer.zero_grad()
    loss.backward()
    online_dqn.optimizer.step()

In [12]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
env = gym.make('CartPole-v1')
input_shape = 4
num_actions = env.action_space.n

In [19]:
online_net = OnlineDQN(input_shape, num_actions).to(DEVICE)

In [24]:
summary(online_net, (8, 512))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (16x512 and 4x512)

In [ ]:
def ddqn():
    """
    """

    pass